# Homework 3 — Clasificación

Solución de `homework.md` usando `data/course_lead_scoring_2026.csv`.

## Dataset

El dataset del deber es de lead scoring: contiene información sobre personas interesadas en una plataforma educativa y se utiliza para predecir si finalmente se registraron o se inscribieron.

La variable objetivo es:

converted: indica si el cliente se convirtió —se registró en la plataforma— o no.
Algunas variables incluyen:

industry: industria del cliente
location: ubicación
lead_source: origen del contacto
employment_status: situación laboral
annual_income: ingreso anual
number_of_courses_viewed: número de cursos vistos
interaction_count: cantidad de interacciones
lead_score: puntuación del prospecto

En resumen, el deber consiste en analizar qué características ayudan a predecir mejor la conversión de potenciales clientes en estudiantes registrados.

In [1]:
import pandas as pd

In [2]:
file = "./data/course_lead_scoring_2026.csv"

In [3]:
df = pd.read_csv(file)

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   lead_source               4852 non-null   str    
 1   industry                  4760 non-null   str    
 2   employment_status         4806 non-null   str    
 3   location                  4792 non-null   str    
 4   annual_income             4631 non-null   float64
 5   number_of_courses_viewed  5000 non-null   int64  
 6   interaction_count         5000 non-null   int64  
 7   lead_score                4965 non-null   float64
 8   converted                 5000 non-null   int64  
dtypes: float64(2), int64(3), str(4)
memory usage: 526.0 KB


In [5]:
df.shape

(5000, 9)

In [6]:
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


## Data preparation

In [7]:
df.isna().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [8]:
categorical = list(df.columns[df.dtypes == "str"])

In [9]:
df[categorical] = df[categorical].fillna("NA")

In [10]:
numerical = [
    "annual_income",
    "number_of_courses_viewed",
    "interaction_count",
    "lead_score",
]

In [11]:
df[numerical] = df[numerical].fillna(0)

In [12]:
df.isna().sum()

lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

## Q1. Most frequent category in `industry`

In [13]:
df["industry"].value_counts()

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64

In [14]:
df["industry"].mode()

0    technology
Name: industry, dtype: str

**Respuesta:** `technology`

## Q2. Correlation matrix (numerical features)

In [15]:
df[numerical].head()

,annual_income,number_of_courses_viewed,interaction_count,lead_score
0,88160.0,3,4,0.64
1,72688.0,3,7,0.72
2,44697.0,3,4,0.58
3,0.0,3,6,0.57
4,24062.0,4,5,0.62


In [16]:
C = df[numerical]

In [17]:
C

,annual_income,number_of_courses_viewed,interaction_count,lead_score
0,88160.0,3,4,0.64
1,72688.0,3,7,0.72
2,44697.0,3,4,0.58
3,0.0,3,6,0.57
4,24062.0,4,5,0.62
...,...,...,...,...
4995,45834.0,2,2,0.37
4996,84831.0,2,5,0.51
4997,53636.0,4,6,0.75
4998,69430.0,4,8,0.79


In [18]:
correlation_matrix = C.corr()

In [19]:
correlation_matrix

,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.000000,0.161300,0.122842,0.229496
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204
interaction_count,0.122842,0.721609,1.000000,0.915746
lead_score,0.229496,0.757204,0.915746,1.000000


**Respuesta:** `interaction_count` y `lead_score` (corr ≈ 0.92)

## Split the data

In [20]:
from sklearn.model_selection import train_test_split

In [21]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

In [22]:
len(df_train), len(df_val), len(df_test)

(3000, 1000, 1000)

In [23]:
df_train.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
4702,paid_ads,finance,employed,asia,65980.0,2,3,0.37,1
2155,events,finance,employed,south_america,71662.0,0,0,0.09,0
289,referral,NA,employed,asia,40220.0,2,5,0.55,1
1822,organic_search,finance,employed,north_america,88352.0,2,2,0.45,0
3442,events,technology,employed,north_america,89316.0,1,5,0.00,1


In [24]:
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

In [25]:
y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

In [26]:
del df_train["converted"]
del df_val["converted"]
del df_test["converted"]

## Q3. Mutual information score (categorical features)

In [27]:
from sklearn.metrics import mutual_info_score

In [28]:
def mutual_info_converted_score(series):
    return mutual_info_score(series, y_train)


mi = df_train[categorical].apply(mutual_info_converted_score)
round(mi.sort_values(ascending=False), 2)

lead_source          0.03
employment_status    0.02
industry             0.00
location             0.00
dtype: float64

**Respuesta:** `lead_source` (MI ≈ 0.03, reduce ~5% de la incertidumbre)

## Q4. Logistic regression (accuracy on validation)

In [29]:
categorical

['lead_source', 'industry', 'employment_status', 'location']

In [30]:
numerical

['annual_income',
 'number_of_courses_viewed',
 'interaction_count',
 'lead_score']

In [31]:
from sklearn.feature_extraction import DictVectorizer

dv = DictVectorizer(sparse=False)

train_dict = df_train[categorical + numerical].to_dict(orient="records")
X_train = dv.fit_transform(train_dict)

val_dict = df_val[categorical + numerical].to_dict(orient="records")
X_val = dv.transform(val_dict)

In [32]:
X_train.shape

(3000, 28)

In [33]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(solver="liblinear", C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multiclass` problems (`n_classes >= 3`), all solvers except 'liblinear' minimize the full multinomial loss, 'liblinear' will raise an error.- 'newton-cholesky' is a good choice for `n_samples` >> `n_features * n_classes`, especially with one-hot encoded categorical features with rare categories. Be aware that the memory usage of this solver has a quadratic dependency on `n_features * n_classes` because it explicitly computes the full Hessian matrix.- For small datasets, 'liblinear' is a good choice, whereas 'sag' and 'saga' are faster for large ones;- 'liblinear' can only handle binary classification by default. To apply a one-versus-rest scheme for the multiclass setting one can wrap it with the :class:`~sklearn.multiclass.OneVsRestClassifier`... warning:: The choice of the algorithm depends on the penalty chosen (`l1_ratio=0` for L2-penalty, `l1_ratio=1` for L1-penalty and `0 < l1_ratio < 1` for Elastic-Net) and on (multinomial) multiclass support: ================= ======================== ====================== solver l1_ratio multinomial multiclass ================= ======================== ====================== 'lbfgs' l1_ratio=0 yes 'liblinear' l1_ratio=1 or l1_ratio=0 no 'newton-cg' l1_ratio=0 yes 'newton-cholesky' l1_ratio=0 yes 'sag' l1_ratio=0 yes 'saga' 0<=l1_ratio<=1 yes ================= ======================== ======================.. note:: 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`... seealso:: Refer to the :ref:`User Guide <Logistic_regression>` for more information regarding :class:`LogisticRegression` and more specifically the :ref:`Table <logistic_regression_solvers>` summarizing solver/penalty supports... versionadded:: 0.17 Stochastic Average Gradient (SAG) descent solver. Multinomial support in version 0.18... versionadded:: 0.19 SAGA solver... versionchanged:: 0.22 The default solver changed from 'liblinear' to 'lbfgs' in 0.22... versionadded:: 1.2 newton-cholesky solver. Multinomial support in version 1.6.",'liblinear'
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The 

In [34]:
y_pred = model.predict_proba(X_val)[:, 1]
converted_decision = y_pred >= 0.5
round(float((y_val == converted_decision).mean()), 2)

0.65

**Respuesta:** 0.65

## Q5. Feature elimination

In [35]:
acc_all_features = (y_val == converted_decision).mean()

In [36]:
to_remove = ["lead_source", "number_of_courses_viewed", "interaction_count"]

In [37]:
for col in to_remove:
    cols = categorical + numerical
    cols.remove(col)
    dv = DictVectorizer(sparse=False)

    train_dict = df_train[cols].to_dict(orient="records")
    X_train = dv.fit_transform(train_dict)

    val_dict = df_val[cols].to_dict(orient="records")
    X_val = dv.transform(val_dict)

    model = LogisticRegression(
        solver="liblinear", C=1.0, max_iter=1000, random_state=42
    )
    model.fit(X_train, y_train) 

    y_pred = model.predict_proba(X_val)[:, 1]
    converted_decision = y_pred >= 0.5
    print(f"Sin {col}: {round(acc_all_features - (y_val == converted_decision).mean(), 4)}")

Sin lead_source: 0.003
Sin number_of_courses_viewed: 0.002
Sin interaction_count: 0.044


**Respuesta:** `number_of_courses_viewed` (diferencia más chica: 0.002)

## Q6. Regularized logistic regression (best C)

In [38]:
for c in [0.000001, 0.00001, 0.0001, 0.001]:
    dv = DictVectorizer(sparse=False)
    
    train_dict = df_train[categorical + numerical].to_dict(orient="records")
    X_train = dv.fit_transform(train_dict)
    
    val_dict = df_val[categorical + numerical].to_dict(orient="records")
    X_val = dv.transform(val_dict)
    
    model = LogisticRegression(
        solver="liblinear", C=c, max_iter=1000, random_state=42
    )
    model.fit(X_train, y_train) 
    
    y_pred = model.predict_proba(X_val)[:, 1]
    converted_decision = y_pred >= 0.5
    print(c, round(float((y_val == converted_decision).mean()), 3))

1e-06 0.598


1e-05 0.598
0.0001 0.613
0.001 0.645


**Respuesta:** C = 0.001 (accuracy ≈ 0.645)